# 🎙️ Pipeline Fallback Step: Merging Speaker Turns: Pandas Vectorized Merging

Refines diarized speaker timelines by calculating shifts and merging segments using vectorized Pandas dataframe operations.

## Environment Setup

In [ ]:
import pandas as pd
import json
import os

## Google Drive Mount & Form Configuration

In [ ]:
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive mount error: {e}")

# @markdown ### 📂 Refinement Configuration
input_json_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/maraulikhurad3_raw_timeline.json" # @param {type:"string"}
output_json_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/maraulikhurad3_refined_timeline.json" # @param {type:"string"}

os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

## Execute Fallback Processing

In [ ]:
# @markdown ### ⏱️ Refinement Parameters
max_merge_gap = 1.5 # @param {type:"number"}

try:
    with open(input_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    df = pd.DataFrame(data)
    # Parse timestamps
    df['start_val'] = df['time'].apply(lambda x: float(x.split('-')[0].replace('[','').strip().split(':')[-2])*60 + float(x.split('-')[0].replace('[','').strip().split(':')[-1]))
    df['end_val'] = df['time'].apply(lambda x: float(x.split('-')[-1].replace(']','').strip().split(':')[-2])*60 + float(x.split('-')[-1].replace(']','').strip().split(':')[-1]))
    
    # Calculate group indicators
    df['prev_speaker'] = df['speaker'].shift(1)
    df['prev_end'] = df['end_val'].shift(1)
    df['new_group'] = (df['speaker'] != df['prev_speaker']) | ((df['start_val'] - df['prev_end']) > max_merge_gap)
    df['group_id'] = df['new_group'].cumsum()
    
    # Group and merge
    merged = df.groupby('group_id').agg({
        'speaker': 'first',
        'start_val': 'min',
        'end_val': 'max',
        'text': lambda x: " ".join(x)
    }).reset_index()
    
    refined_entries = []
    for _, row in merged.iterrows():
        refined_entries.append({
            "time": f"[{int(row['start_val']//60):02d}:{int(row['start_val']%60):02d} - {int(row['end_val']//60):02d}:{int(row['end_val']%60):02d}]",
            "speaker": row['speaker'],
            "text": row['text']
        })
        
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(refined_entries, f, indent=4)
    print(f"[SUCCESS] Refined timeline saved to: {output_json_path}")
except Exception as e:
    print(f"[ERROR] Pandas grouping failed: {e}")